# Лабораторная работа №1. Пункты 7-9

Вход: очищенный и токенизированный корпус `body_clean` из `1_eda.ipynb` (`artifacts/train_clean.csv`, `artifacts/test_clean.csv`). Токенизация — по пробелу: тексты уже содержат `<s>`, `</s>` и технические токены (`[url]`, `[num]`, `[hex]`, `[code]`, `[user]`, `[issueref]`, `[nonlatin]`) как единые "слова".

В этом ноутбуке:
- п.7 — построение биграммной и триграммной статистической языковой модели со словарём и обработкой `[unk]`;
- п.8 — сглаживание (Add-k/Лидстоун), функция перплексии через log-likelihood, сравнение со сглаживанием и без;
- п.9 — генерация 5 текстов Марковской цепью и качественная оценка.


In [1]:
import math
import random
import time
from collections import Counter, defaultdict

import pandas as pd

ARTIFACTS_DIR = "artifacts"
train = pd.read_csv(f"{ARTIFACTS_DIR}/train_clean.csv")
test = pd.read_csv(f"{ARTIFACTS_DIR}/test_clean.csv")

train_tokens_docs = [s.split() for s in train["body_clean"]]
test_tokens_docs = [s.split() for s in test["body_clean"]]
print(f"train: {len(train_tokens_docs):,} документов, test: {len(test_tokens_docs):,} документов")


train: 40,000 документов, test: 10,000 документов


## Шаг 7. Словарь модели, обработка [unk] и N-граммная модель

In [ ]:
MIN_COUNT = 2

train_freq = Counter()
for doc in train_tokens_docs:
    train_freq.update(doc)

print(f"Токенов уникальных до фильтрации по частоте: {len(train_freq):,}")

VOCAB = {w for w, c in train_freq.items() if c >= MIN_COUNT} | {"<s>", "</s>", "[unk]"}
print(f"Итоговый словарь модели (частота >= {MIN_COUNT}, плюс <s>/</s>/[unk]): {len(VOCAB):,} токенов")

def replace_oov(tokens):
    '''Заменяет все слова, не входящие в словарь модели, на [unk].'''
    return [t if t in VOCAB else "[unk]" for t in tokens]

train_seqs = [replace_oov(doc) for doc in train_tokens_docs]
test_seqs = [replace_oov(doc) for doc in test_tokens_docs]

unk_share_test = sum(tok.count("[unk]") for tok in test_seqs) / sum(len(tok) for tok in test_seqs)
print(f"Доля токенов [unk] в test после подстановки: {unk_share_test:.2%}")


Токенов уникальных до фильтрации по частоте: 110,139
Итоговый словарь модели (частота >= 2, плюс <s>/</s>/[unk]): 49,415 токенов


Доля токенов [unk] в test после подстановки: 5.29%

In [3]:
def pad(tokens, n):
    '''Дополняет последовательность (n-1) токенами <s> для корректного построения
    контекста самых первых n-грамм (в очищенном тексте уже есть один <s>, здесь
    гарантируется, что перед первым словом окажется ровно n-1 токенов начала).'''
    return ["<s>"] * (n - 1) + tokens[1:]

def build_ngram_counts(seqs, n):
    '''Считает частоты контекстов длины (n-1) и самих n-грамм (context, word).'''
    context_counts = Counter()
    ngram_counts = Counter()
    for tokens in seqs:
        padded = pad(tokens, n)
        for i in range(n - 1, len(padded)):
            context = tuple(padded[i - n + 1:i])
            word = padded[i]
            context_counts[context] += 1
            ngram_counts[(context, word)] += 1
    return context_counts, ngram_counts

t0 = time.time()
bigram_context_counts, bigram_counts = build_ngram_counts(train_seqs, 2)
print(f"Биграммная модель: {len(bigram_context_counts):,} контекстов, "
      f"{len(bigram_counts):,} уникальных биграмм ({time.time() - t0:.1f} сек)")

t0 = time.time()
trigram_context_counts, trigram_counts = build_ngram_counts(train_seqs, 3)
print(f"Триграммная модель: {len(trigram_context_counts):,} контекстов, "
      f"{len(trigram_counts):,} уникальных триграмм ({time.time() - t0:.1f} сек)")

V = len(VOCAB)


Биграммная модель: 49,414 контекстов, 747,584 уникальных биграмм (2.0 сек)


Триграммная модель: 740,868 контекстов, 1,620,795 уникальных триграмм (2.5 сек)


## Шаг 8. Сглаживание и перплексия

**Метод сглаживания:** реализуется **Add-k (обобщённое сглаживание Лидстоуна)** — при `k=1` это классическое сглаживание Лапласа, при `k<1` — более мягкий вариант:

\[
P(w \mid c) = \frac{\text{count}(c, w) + k}{\text{count}(c) + k \cdot |V|}
\]

Ниже сравниваются оба крайних случая (Лаплас, `k=1`) и подобранный меньший `k`, чтобы показать, как выбор `k` влияет на итоговую перплексию при большом словаре.

In [ ]:
def log_prob_sequence(tokens, n, context_counts, ngram_counts, smoothing=None, k=1.0):
    '''Сумма log P(w_i | context) по последовательности. Возвращает (None, None),
    если хотя бы одна n-грамма без сглаживания имеет нулевую вероятность —
    это явный сигнал 'perplexity не определена' вместо математической ошибки log(0).'''
    padded = pad(tokens, n)
    total_log_prob = 0.0
    n_predictions = 0
    for i in range(n - 1, len(padded)):
        context = tuple(padded[i - n + 1:i])
        word = padded[i]
        c_context = context_counts.get(context, 0)
        c_ngram = ngram_counts.get((context, word), 0)

        if smoothing == "addk":
            prob = (c_ngram + k) / (c_context + k * V)
        else:
            if c_context == 0 or c_ngram == 0:
                return None, None
            prob = c_ngram / c_context

        total_log_prob += math.log(prob)
        n_predictions += 1
    return total_log_prob, n_predictions


def perplexity(seqs, n, context_counts, ngram_counts, smoothing=None, k=1.0):
    '''Perplexity = exp(-mean(log P)). Возвращает (perplexity, доля документов
    с неопределённой (нулевой) вероятностью хотя бы одной n-граммы).'''
    total_log_prob = 0.0
    total_predictions = 0
    undefined_docs = 0
    for tokens in seqs:
        log_prob, n_pred = log_prob_sequence(tokens, n, context_counts, ngram_counts, smoothing, k)
        if log_prob is None:
            undefined_docs += 1
            continue
        total_log_prob += log_prob
        total_predictions += n_pred

    undefined_share = undefined_docs / len(seqs)
    if total_predictions == 0:
        return float("inf"), undefined_share
    return math.exp(-total_log_prob / total_predictions), undefined_share


In [5]:
results = []
for n, ctx_counts, ngram_counts, label in [
    (2, bigram_context_counts, bigram_counts, "bigram"),
    (3, trigram_context_counts, trigram_counts, "trigram"),
]:
    pp_train_ns, und_train_ns = perplexity(train_seqs, n, ctx_counts, ngram_counts, smoothing=None)
    pp_test_ns, und_test_ns = perplexity(test_seqs, n, ctx_counts, ngram_counts, smoothing=None)
    pp_test_laplace, und_test_laplace = perplexity(test_seqs, n, ctx_counts, ngram_counts, smoothing="addk", k=1.0)
    pp_test_addk, und_test_addk = perplexity(test_seqs, n, ctx_counts, ngram_counts, smoothing="addk", k=0.01)

    results.append({"n-gram": label, "выборка": "train", "сглаживание": "без сглаживания",
                     "perplexity": round(pp_train_ns, 2), "% документов с неопр. вероятностью": f"{und_train_ns:.1%}"})
    results.append({"n-gram": label, "выборка": "test", "сглаживание": "без сглаживания",
                     "perplexity": "не определена (см. текст ниже)" if und_test_ns > 0 else round(pp_test_ns, 2),
                     "% документов с неопр. вероятностью": f"{und_test_ns:.1%}"})
    results.append({"n-gram": label, "выборка": "test", "сглаживание": "add-k, k=1 (Лаплас)",
                     "perplexity": round(pp_test_laplace, 2), "% документов с неопр. вероятностью": f"{und_test_laplace:.1%}"})
    results.append({"n-gram": label, "выборка": "test", "сглаживание": "add-k, k=0.01",
                     "perplexity": round(pp_test_addk, 2), "% документов с неопр. вероятностью": f"{und_test_addk:.1%}"})

results_df = pd.DataFrame(results)
results_df


,n-gram,выборка,сглаживание,perplexity,% документов с неопр. вероятностью
0,bigram,train,без сглаживания,91.47,0.0%
1,bigram,test,без сглаживания,не определена (см. текст ниже),90.9%
2,bigram,test,"add-k, k=1 (Лаплас)",2271.54,0.0%
3,bigram,test,"add-k, k=0.01",528.02,0.0%
4,trigram,train,без сглаживания,7.39,0.0%
5,trigram,test,без сглаживания,не определена (см. текст ниже),94.1%
6,trigram,test,"add-k, k=1 (Лаплас)",13764.6,0.0%
7,trigram,test,"add-k, k=0.01",3142.02,0.0%


Выводы:
- доля документов test с хотя бы одной неопределённой (нулевой) вероятностью n-граммы у триграммной модели выше, чем у биграммной (см. столбец "% документов с неопр. вероятностью" для строк "без сглаживания") — триграммные контексты значительно более разреженные, и на новых данных они гораздо чаще не встречались в train.

- без сглаживания perplexity на test математически **не определена** для подавляющего большинства документов — как только модель встречает n-грамму с нулевым count, log(0) уходит в `-inf`, и итоговая perplexity всего документа не определена. То число, которое всё-таки выводится в строке "без сглаживания" для train, — не показатель качества модели на новых данных, это переобучение (модель считает вероятность по тем же данным, на которых её строили, поэтому там undefined = 0%). Со сглаживанием (`add-k`) вероятность любой n-граммы становится строго положительной, поэтому perplexity определена для 100% документов test — но платой за это является более высокое абсолютное значение perplexity, особенно при `k=1` (Лаплас): при большом словаре модели (десятки тысяч токенов, точное значение `|V|` см. в выводе кода из шага 7) слагаемое `k*|V|` в знаменателе становится доминирующим и "размазывает" вероятностную массу почти равномерно по всему словарю, из-за чего perplexity получается завышенной. Уменьшение `k` (Lidstone, `k=0.01`) даёт заметно меньшую (более реалистичную) perplexity, оставаясь корректно определённой на 100% документов — это и есть математически обоснованный аргумент в пользу сглаживания и подбора его силы (`k`), а не просто наличия сглаживания как такового.

## Шаг 9. Генерация текста Марковской цепью

In [6]:
def build_followers(seqs, n):
    '''Для каждого контекста длины (n-1) собирает распределение следующих слов
    (нужно для генерации — сэмплирования по цепи Маркова).'''
    followers = defaultdict(Counter)
    for tokens in seqs:
        padded = pad(tokens, n)
        for i in range(n - 1, len(padded)):
            context = tuple(padded[i - n + 1:i])
            word = padded[i]
            followers[context][word] += 1
    return followers

bigram_followers = build_followers(train_seqs, 2)
trigram_followers = build_followers(train_seqs, 3)

unigram_dist = Counter()
for tokens in train_seqs:
    unigram_dist.update(t for t in tokens if t != "<s>")


def generate_text(n, followers, backoff_followers=None, max_len=40, seed=None):
    '''Генерация методом Марковской цепи: на каждом шаге сэмплируем следующее
    слово из распределения P(w | context), context = последние (n-1) токенов.
    Если контекст не встречался в train (для триграммы это возможно уже после
    нескольких сгенерированных слов), делаем backoff к модели меньшего порядка,
    а если и там пусто -- к общему юниграммному распределению.
    Останавливаемся на </s> или при достижении max_len.'''
    rng = random.Random(seed)
    history = ["<s>"]
    hit_end = False
    for _ in range(max_len):
        context = tuple((["<s>"] * (n - 1) + history)[-(n - 1):])
        dist = followers.get(context)
        if not dist and backoff_followers is not None:
            dist = backoff_followers.get(context[-1:])
        if not dist:
            dist = unigram_dist
        words, weights = zip(*dist.items())
        next_word = rng.choices(words, weights=weights, k=1)[0]
        history.append(next_word)
        if next_word == "</s>":
            hit_end = True
            break
    return " ".join(history), hit_end


print("=== 5 текстов, триграммная модель (с backoff к биграммной при неизвестном контексте) ===\n")
generated_texts = []
for seed in range(5):
    text, hit_end = generate_text(3, trigram_followers, backoff_followers=bigram_followers, max_len=40, seed=seed)
    generated_texts.append((text, hit_end))
    stop_reason = "остановка по </s>" if hit_end else "остановка по max_len (40 токенов)"
    print(f"[{seed + 1}] ({stop_reason})\n{text}\n")


=== 5 текстов, триграммная модель (с backoff к биграммной при неизвестном контексте) ===

[1] (остановка по max_len (40 токенов))
<s> bot ragnarok [unk] ind server chaos loki demo video [url] url isn t and you can erase them or do i put on this means you should save as kibana pane [num] type [unk] data before with this search result

[2] (остановка по max_len (40 токенов))
<s> i haven t done and display the annotated image on server s url like [url] now you should just do this python error [unk] error when i try to add a column [url] dashboard settings for it there s a

[3] (остановка по </s>)
<s> href resources css style css </s>

[4] (остановка по </s>)
<s> the gradle based spring boot admin console using nightly js w2ui js v judcyvehtw6kqtpts3ehma2 [num] at the root of your json file content with the same operations on this module [url] as i scroll in visually the ui </s>

[5] (остановка по </s>)
<s> the error as it is showing through the holes of the past few months ago has the doc

**Гипотеза H10 и качественная оценка:** локально (на уровне соседних 2-3 слов) сгенерированные тексты выглядят синтаксически правдоподобно — модель воспроизводит типичные для GitHub Issues словосочетания и обороты ("i haven t done", "there s a", "the error as it is showing", "you should just do this"), потому что именно такие короткие цепочки часто встречались в train. Но на уровне всего сгенерированного текста связного смысла нет: тема "уплывает" от предложения к предложению (например, переход от обсуждения ошибки к упоминанию dashboard/json/gradle в одном и том же тексте), потому что триграммная модель учитывает только 2 предыдущих слова и не обладает памятью о теме или синтаксическом дереве всего документа. Это ожидаемое и объяснимое ограничение N-граммных Марковских моделей, а не ошибка реализации — гипотеза H10 подтверждается.

## Выводы по ноутбуку 3

1. Построена биграммная и триграммная модели со словарём train (порог по частоте) и единым токеном `[unk]` для редких/незнакомых слов, включая OOV-слова из test.
2. Реализована функция perplexity через сумму log-likelihood (без переполнения), которая явно сигнализирует о математически неопределённых случаях (нулевая вероятность), а не пытается их замолчать.
3. Показано, что триграммная модель без сглаживания страдает от разреженности данных сильнее биграммной (H8), а сглаживание Add-k необходимо, чтобы perplexity на test вообще была определена (H9); при этом величина `k` заметно влияет на итоговое значение perplexity при большом словаре.
4. Сгенерированные Марковской цепью тексты локально грамматичны, но не обладают связностью на уровне всего документа — типичное и объяснимое ограничение N-граммных моделей (H10).
